# Training on Colab

Runtime → Change runtime type → T4 GPU.

## Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/DeepLearning-Project'
!mkdir -p "$DRIVE_DIR/outputs"

In [ ]:
%cd /content
!test -d DeepLearning-Project || git clone -q https://github.com/Moazdarwishh/DeepLearning-Project.git
%cd /content/DeepLearning-Project
!git pull -q
!git log --oneline -3

In [ ]:
!pip install -q datasets mlflow imagehash pyspellchecker

In [ ]:
!rm -rf outputs && ln -s "$DRIVE_DIR/outputs" outputs
!test -f "$DRIVE_DIR/mlflow.db" && cp "$DRIVE_DIR/mlflow.db" mlflow.db
!python -m pytest -q

## Quick check

In [ ]:
!python -m src.cli train --task classification --config configs/cls_scratch.yaml \
    --set train.epochs=2 train.limit_train_batches=20 train.limit_val_batches=5 \
          tracking.backend=none experiment.name=smoke-colab

## Training

In [ ]:
!python -m src.cli train --task classification --config configs/cls_scratch.yaml

In [ ]:
!python -m src.cli train --task classification --config configs/cls_resnet50_probe.yaml

In [ ]:
!python -m src.cli train --task classification --config configs/cls_resnet50.yaml

Resume an interrupted run:

In [ ]:
RUN = 'cls-scratch'
!python -m src.cli train --task classification --config outputs/runs/$RUN/config.yaml \
    --resume outputs/runs/$RUN/last.pt

## Hyperparameter search

In [ ]:
!python -m src.cli tune --task classification --config configs/tune_grid.yaml

In [ ]:
!python -m src.cli tune --task classification --config configs/tune_random.yaml

In [ ]:
!python -m src.cli tune --task classification --compare outputs/tuning/tune-grid outputs/tuning/tune-random
from IPython.display import Image, display
display(Image('outputs/tuning/comparison.png'))

## Validation results

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('outputs/runs/cls-*/summary.json')):
    s = json.load(open(f)); v = s['best_val']
    rows.append({'run': s['run'], 'best_epoch': s['best_epoch'], 'minutes': s['train_time_min'],
                 'val_acc': v['val/accuracy'], 'val_top5': v['val/top5_accuracy'], 'val_macro_f1': v['val/macro_f1']})
pd.DataFrame(rows).round(4)

In [ ]:
for f in sorted(glob.glob('outputs/runs/cls-*/curves.png')):
    print(f); display(Image(f))

## Test results

In [ ]:
RUNS = ['cls-scratch', 'cls-resnet50-probe', 'cls-resnet50-ft']
for run in RUNS:
    !python -m src.cli evaluate --task classification --checkpoint outputs/runs/$run/best.pt --split test

In [ ]:
run_dirs = ' '.join(f'outputs/runs/{r}' for r in RUNS)
!python -m src.cli compare --task classification --runs $run_dirs --split test
display(Image('outputs/comparison_test.png'))

In [ ]:
!python -m src.cli gradcam --task classification --split test --num-samples 6 \
    --checkpoint outputs/runs/cls-resnet50-ft/best.pt outputs/runs/cls-scratch/best.pt
for f in sorted(glob.glob('outputs/gradcam/*/*.png')):
    print(f); display(Image(f))

## Save MLflow database

In [ ]:
!cp mlflow.db "$DRIVE_DIR/mlflow.db"